# Inflasi awal tahun: siapa yang naik?

Setiap Januari terdengar keluhan yang sama: "awal tahun serba mahal".
Saya penasaran seberapa benar keluhan itu: apakah Januari memang bulan paling
mahal sepanjang tahun, dan kalau ada yang naik, "siapa" tepatnya di keranjang
belanja yang menaikkannya?

In [ ]:
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd

from tools import gaya

gaya.terapkan()

# Lapis A: indeks CPI bulanan (sumber nasional BPS, relay statistik resmi OECD)
ihk = pd.read_csv("data/mentah/oecd/cpi-idn-indeks-bulanan.csv")[["TIME_PERIOD", "OBS_VALUE"]]
ihk.columns = ["bln", "ihk"]
ihk["bln"] = pd.to_datetime(ihk["bln"])
ihk = ihk.set_index("bln")["ihk"].sort_index()
mom = (ihk.pct_change() * 100).dropna()

# Lapis B: harga dapur harian PIHPS nasional, satu snapshot pertengahan per bulan
ph = pd.read_csv("data/harga-nasional-bulanan.csv", parse_dates=["tanggal"])

print("CPI:", ihk.index.min().date(), "->", ihk.index.max().date(), f"({len(ihk)} bulan)")
print("PIHPS:", ph.komoditas.nunique(), "komoditas,", ph.tanggal.nunique(), "bulan unik")

CPI: 2015-01-01 -> 2026-08-01 (140 bulan)
PIHPS: 10 komoditas, 80 bulan unik


/tmp/ipykernel_23381/3518971619.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ihk["bln"] = pd.to_datetime(ihk["bln"])


## Data & cara ukur

Dua lapis sumber yang sengaja dipisah peran. **Lapis resmi:** indeks harga
konsumen bulanan (IHK) yang dihitung BPS dan direlay lewat API statistik resmi
OECD tanpa kunci; 140 bulan, Januari 2015 sampai Agustus 2026. Yang dibaca di
sini bukan besaran harga, melainkan perubahan bulan ke bulan (inflasi bulanan),
pola musimnya dirata-rata per kalender bulan. Catatan jujur: katalog BPS tidak
bisa diakses dari lingkungan panen ini (tabel membalas 403, API minta token),
jadi dipakai relay resmi OECD dengan angka yang sama.

**Lapis dapur:** PIHPS (bi.go.id/hargapangan) memantau harga eceran sepuluh
komoditas pangan strategis di pasar tradisional seluruh provinsi tiap hari
kerja; diambil satu snapshot pertengahan bulan, Oktober 2018 sampai Agustus
2026. "Awal tahun" dibaca sebagai perubahan Desember pertengahan ke Januari
pertengahan. Kontrak sumber, sembilan bulan kosong dari server, dan batasnya
diwartakan di `data/README.md`.

In [2]:
# cakupan dasar lapis resmi
ringk_bln = mom.groupby(mom.index.month).agg(["mean", "median", "std", "count"])
ringk_bln.index = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]
print(ringk_bln.round(3).sort_values("median", ascending=False))
tahun_puncak_jan = sum(
    mom.loc[mom.index.year == t].idxmax().month == 1
    for t in range(2015, 2026) if mom.loc[mom.index.year == t].size == 12)
print("Januari jadi bulan tertinggi:", tahun_puncak_jan, "tahun dari",
      sum(1 for t in range(2015, 2026) if mom.loc[mom.index.year == t].size == 12))

      mean  median    std  count
Des  0.565   0.570  0.179     11
Jun  0.361   0.488  0.296     12
Jan  0.273   0.325  0.453     11
Nov  0.261   0.268  0.114     11
Mei  0.231   0.257  0.272     12
Jul  0.269   0.246  0.345     12
Apr  0.297   0.193  0.422     12
Mar  0.354   0.185  0.452     12
Feb  0.080   0.131  0.316     12
Okt  0.089   0.075  0.127     11
Agu  0.019  -0.023  0.156     12
Sep  0.110  -0.038  0.387     11
Januari jadi bulan tertinggi: 2 tahun dari 10


## Pembedahan

Gambar pertama: pola musim inflasi bulanan nasional. Kalau desas-desus "awal
tahun serba mahal" itu benar, Januari harusnya duduk di puncak daftar ini.

In [3]:
BULAN_ID = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Puncaknya bukan Januari: justru Desember bulan paling mahal",
        "Perubahan bulanan harga (rerata 2015-2026; titik: tiap tahun). Januari hanya "
        "peringkat kelima, di bawah Desember, Juni, Maret, dan April.",
        "CPI bulanan nasional (BPS via relay resmi OECD), 140 bulan 2015-2026",
        int(len(ihk)),
    )
    xs = np.arange(12)
    for b in range(1, 13):
        titik = mom.loc[mom.index.month == b]
        if len(titik):
            ax.scatter(np.full(len(titik), b - 1) + np.random.default_rng(0).normal(0, 0.07, len(titik)),
                       titik.values, s=10, color=gaya.INK2, alpha=0.35, zorder=2)
    rata = [mom.loc[mom.index.month == b].mean() for b in range(1, 13)]
    warna = [gaya.AKSEN if b in (0, 11) else gaya.INK for b in range(12)]
    ax.bar(xs, rata, color=warna, alpha=0.85, width=0.62, zorder=3)
    ax.axhline(0, color=gaya.INK2, lw=0.9)
    ax.annotate(f"Des: {rata[11]:.2f}% per bulan\ntertinggi rerata".replace(",", "."),
                xy=(11, rata[11]), xytext=(6.8, rata[11] + 0.05), fontsize=8.8 * fs, color=gaya.INK,
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9), ha="left")
    ax.annotate(f"Jan: {rata[0]:.2f}%\nperingkat 5".replace(",", "."),
                xy=(0, rata[0]), xytext=(1.2, -0.28), fontsize=8.8 * fs, color=gaya.INK,
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9))
    ax.set_xticks(xs); ax.set_xticklabels(BULAN_ID, fontsize=9 * fs)
    ax.set_ylabel("inflasi bulanan (% per bulan, rerata)", fontsize=9.5 * fs)
    ax.grid(True, axis="y"); ax.grid(False, axis="x")
    gaya.simpan(fig, "01-musim-inflasi", mode)


for mode in gaya.MODE:
    grafik_1(mode)

Jadi di indeks resmi, Januari bukan puncak. Tapi keranjang belanja bukan
indeks tertimbang; yang terasa adalah daftar bahan. Gambar kedua membaca
sepuluh bahan dapur di setiap awal tahun: yang berada di atas nol adalah yang
konsisten naik lebih tinggi biasa, yang di bawah nol malah turun.

In [4]:
def transisi(kom, d):
    d = d.set_index("tanggal")["harga_nasional"].sort_index()
    d.index = d.index.to_period("M")
    d = d[~d.index.duplicated(keep="first")]
    rows = []
    for (pm, pv), (cm, cv) in zip(zip(d.index[:-1], d.values[:-1]), zip(d.index[1:], d.values[1:])):
        if (cm.year - pm.year) * 12 + (cm.month - pm.month) == 1:
            rows.append({"kom": kom, "ke": str(cm), "dari_bln": pm.month, "delta": cv / pv - 1})
    return pd.DataFrame(rows)

TRANS = pd.concat([transisi(k, d) for k, d in ph.groupby("komoditas")], ignore_index=True)
TRANS["januari"] = TRANS["dari_bln"] == 12
JAN = TRANS[TRANS.januari]
N_JAN = int(JAN.ke.nunique())

from numpy.random import default_rng
rng = default_rng(2026)


def ci_med(x):
    x = np.asarray(x, float)
    sm = np.array([np.median(rng.choice(x, len(x))) for _ in range(10000)])
    return float(np.median(x)), float(np.percentile(sm, 2.5)), float(np.percentile(sm, 97.5))


UP = []
for kom, d in TRANS.groupby("kom"):
    at = d.loc[d.januari, "delta"].values
    bl = d.loc[~d.januari, "delta"].values
    m_at = float(np.median(at)); m_bl = float(np.median(bl))
    up, lo, hi = ci_med((at - m_bl))
    naik_kali = int((at > 0).sum())
    UP.append({"kom": kom, "jan": m_at, "land": m_bl, "up": up, "lo": lo, "hi": hi,
               "naik_kali": naik_kali, "n": len(at)})
UP = pd.DataFrame(UP).sort_values("up", ascending=True)
print(f"{len(UP)} komoditas; Januari terbaca: {N_JAN} tahun")
print(UP.assign(jan=lambda d: (d["jan"]*100).round(2), land=lambda d: (d["land"]*100).round(2),
                up=lambda d: (d["up"]*100).round(2)).to_string(index=False))

def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Di dapur: hanya bawang putih yang rutin naik; telur ayam malah turun",
        "Selisih kenaikan awal tahun vs landasan bulan biasa per bahan, tujuh Januari. "
        "Batang penuh warna: selisihnya tidak menembus nol (95% CI bootstrap).",
        f"PIHPS nasional, 10 komoditas x {N_JAN} Januari (2019-2026, Jan 2022 tak ada rilis)",
        int(len(TRANS)),
    )
    df_ = UP.iloc[::-1]
    ys = np.arange(len(df_))
    tegas = (df_["lo"] > 0) | (df_["hi"] < 0)
    BATAS_X = 14.0
    for i, (_, r) in enumerate(df_.iterrows()):
        c = gaya.AKSEN if tegas.iloc[i] else gaya.INK2
        ax.barh(i, r["up"] * 100, color=c, alpha=0.85, height=0.6, zorder=3)
        lo, hi = r["lo"] * 100, r["hi"] * 100
        lo_c, hi_c = max(lo, -BATAS_X), min(hi, BATAS_X)
        ax.plot([lo_c, hi_c], [i, i], color=gaya.INK, lw=1.3, zorder=4)
        if lo < -BATAS_X:
            ax.text(-BATAS_X, i, "◀", va="center", ha="center", fontsize=8 * fs, color=gaya.INK)
        if hi > BATAS_X:
            ax.text(BATAS_X, i, "▶", va="center", ha="center", fontsize=8 * fs, color=gaya.INK)
    ax.axvline(0, color=gaya.INK2, lw=0.9, ls=":")
    ax.set_yticks(ys)
    ax.set_yticklabels([f"{r.kom} ({int(r.naik_kali)}/{int(r.n)})" for _, r in df_.iterrows()],
                       fontsize=9 * fs)
    ax.set_xlim(-BATAS_X * 1.18, BATAS_X * 1.18)
    ax.set_xlabel("selisih kenaikan awal tahun vs landasan (poin persen); "
                  "panah: selang cabai lebih lebar dari frame", fontsize=9.2 * fs)
    ax.grid(True, axis="x"); ax.grid(False, axis="y")
    gaya.simpan(fig, "02-ramp-dapur-januari", mode)


for mode in gaya.MODE:
    grafik_2(mode)

10 komoditas; Januari terbaca: 7 tahun
          kom   jan  land    up        lo        hi  naik_kali  n
  Cabai Merah  0.59  3.88 -3.29 -0.320519  0.400741          4  7
   Telur Ayam -1.71  0.19 -1.90 -0.027538 -0.001938          1  7
  Daging Ayam  0.00  0.58 -0.58 -0.022652  0.027481          3  7
  Daging Sapi  0.08  0.08  0.00 -0.002322  0.006324          4  7
   Gula Pasir  0.28  0.00  0.28  0.000000  0.016173          5  7
        Beras  0.68  0.31  0.37 -0.003125  0.005386          5  7
Minyak Goreng  0.69  0.22  0.48 -0.004850  0.010132          5  7
 Bawang Merah  5.48  4.20  1.29 -0.155472  0.071644          4  7
 Bawang Putih  1.50  0.11  1.39  0.002248  0.071609          6  7
  Cabai Rawit 10.79  1.07  9.72 -0.238500  0.450085          4  7


## Temuan

> Mitos "awal tahun serba mahal" tidak tercermin di indeks resmi: puncak inflasi bulanan rerata ada di Desember (0,57% per bulan), sementara Januari hanya peringkat kelima (0,27%). Di keranjang dapur pun tidak semua naik: dari sepuluh bahan, hanya bawang putih yang konsisten naik melebihi biasa di awal tahun (+1,39 poin persen, 6 dari 7 Januari), dan telur ayam justru rutin turun (-1,90 poin persen, hanya 1 dari 7). Sisanya bergantung tahun: cabai rawit bisa meledak +11% atau datar.

In [5]:
metrik = {
    "lapis_resmi": {
        "n_bulan": int(len(ihk)),
        "median_bulanan_per_kalender": {BULAN_ID[b-1]: round(float(mom.loc[mom.index.month == b].median()), 3) for b in range(1, 13)},
        "rerata_bulanan_per_kalender": {BULAN_ID[b-1]: round(float(mom.loc[mom.index.month == b].mean()), 3) for b in range(1, 13)},
        "januari_puncak_dari_11_tahun": tahun_puncak_jan,
        "januari_2025": round(float(mom.loc[mom.index == pd.Timestamp("2025-01-01")].iloc[0]), 3),
    },
    "lapis_dapur": UP[["kom", "jan", "land", "up", "lo", "hi", "naik_kali", "n"]]
        .assign(jan=lambda d: (d["jan"]*100).round(3), land=lambda d: (d["land"]*100).round(3),
                up=lambda d: (d["up"]*100).round(3), lo=lambda d: (d["lo"]*100).round(3),
                hi=lambda d: (d["hi"]*100).round(3)).to_dict("records"),
    "n_januari_terbaca": N_JAN,
}
print(json.dumps(metrik, indent=2, ensure_ascii=False))
Path("data/metrik-007.json").write_text(json.dumps(metrik, indent=2, ensure_ascii=False))

{
  "lapis_resmi": {
    "n_bulan": 140,
    "median_bulanan_per_kalender": {
      "Jan": 0.325,
      "Feb": 0.131,
      "Mar": 0.185,
      "Apr": 0.193,
      "Mei": 0.257,
      "Jun": 0.488,
      "Jul": 0.246,
      "Agu": -0.023,
      "Sep": -0.038,
      "Okt": 0.075,
      "Nov": 0.268,
      "Des": 0.57
    },
    "rerata_bulanan_per_kalender": {
      "Jan": 0.273,
      "Feb": 0.08,
      "Mar": 0.354,
      "Apr": 0.297,
      "Mei": 0.231,
      "Jun": 0.361,
      "Jul": 0.269,
      "Agu": 0.019,
      "Sep": 0.11,
      "Okt": 0.089,
      "Nov": 0.261,
      "Des": 0.565
    },
    "januari_puncak_dari_11_tahun": 2,
    "januari_2025": -0.758
  },
  "lapis_dapur": [
    {
      "kom": "Cabai Merah",
      "jan": 0.588,
      "land": 3.882,
      "up": -3.294,
      "lo": -32.052,
      "hi": 40.074,
      "naik_kali": 4,
      "n": 7
    },
    {
      "kom": "Telur Ayam",
      "jan": -1.709,
      "land": 0.194,
      "up": -1.903,
      "lo": -2.754,
      "hi":

2471

## Batas & cara penggunaan
Enam hal perlu dibilang. Pertama, lapis resmi direlay lewat OECD; angka adalah angka BPS, tetapi akses langsung ke katalog BPS tertutup dari lingkungan panen, sehingga kontrak asli tidak dapat didokumentasikan ulang di sini. Kedua, INDEKS resmi berganti tahun dasar sepanjang 2015-2026; yang dibaca adalah perubahan bulanannya, yang masih sah rantai. Ketiga, PIHPS memantau sepuluh komoditas pangan di pasar tradisional, bukan komponen resmi IHK: komponen resmi tidak terbuka, jadi "daftar naik" di edisi ini adalah daftar keranjang dapur. Keempat, nasional PIHPS adalah rerata provinsi dihitung server, bukan rerata tertimbang populasi. Kelima, Januari 2022 tidak punya rilis PIHPS (satu dari sembilan bulan kosong server), jadi ramp dibaca dari tujuh Januari, bukan delapan. Keenam, pertengahan bulan ke pertengahan bulan adalah proksi transisi bulan; tidak seluruh gerakan harga "akhir tahun" terwakili hari per hari.

```bash
python3 tools/data_scraping.py     # PIHPS 10 komoditas + CPI OECD (butuh internet)
python3 -m nbconvert --to notebook --execute --inplace analisis.ipynb
```